
# Contents
### 1. The Iris CubeList and Cube objects
#### 1.1 Dask arrays
### 2. Plotting with iris
#### 2.1 Regional example
#### 2.2 Global example
---

# 1. The Iris CubeList and Cube objects

- NetCDF file provided contains air temperature and vertical velocity diagnostics on the hour for a 10-hour LFRic CRM simulation
- The dataset has coordinates (time, z, y, x) with shape (10, 141, 128, 128)

In [ ]:
import warnings
import iris
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore",module="iris.coords")
iris.FUTURE.date_microseconds = True

In [ ]:
# Download dataset from astro webserver
import os
if not os.path.exists("crm_cbl.nc"):
    os.system("curl -O https://www.star.bris.ac.uk/acorbett/devgroup/lfric_iris_intro/crm_cbl.nc")

In [ ]:
# Load the dataset
cubes = iris.load('crm_cbl.nc')

# If multiple diagnostics are present in the file, iris.load() will return a CubeList object containing multiple Cube objects.
# Each cube represents a different diagnostic and has its own metadata and data array.

In [ ]:
# Display the CubeList
cubes

In [ ]:
# Print the coordinates of the air_temperature cube
cubes[0].coords()

## 1.1 Dask arrays

In [ ]:
# Check if the cubes have lazy data (i.e., data that has not been loaded into memory yet)
for cube in cubes:
    print(f"Cube '{cube.name()}' has lazy data: {cube.has_lazy_data()}")

In [ ]:
# Access the air_temperature cube's dask array object via cube.core_data()
print(cubes[0].core_data())

In [ ]:
# We see that shape and chunksize match, meaning our array isn't big enough for dask to consider it worth chunking
# So let's force it to chunk by setting a small chunksize (this in inefficent but is just for demonstration purposes!)

temp_cube = cubes[0].copy() 
temp_cube.data = temp_cube.core_data().rechunk(chunks=(2, 141, 128, 128)) # 2 time dimensions per chunk
print(temp_cube.core_data())

# Now we'll display the task graph to see this chunking in action
temp_cube.core_data().visualize(filename=None)

In [ ]:
# Now we'll go back to the original air_temperature cube and operate on it by taking a vertical mean, for example
# this is done using the cube.collapsed() method, which collapses the cube along a specified dimension using a specified operation (in this case, mean)
vertical_mean = cubes[0].collapsed('height', iris.analysis.MEAN)

# Now take a mean in x
temp_mean_cube = vertical_mean.collapsed('x', iris.analysis.MEAN)
# And take a mean in y
temp_mean_cube = temp_mean_cube.collapsed('y', iris.analysis.MEAN)
# this leaves us with full domain averages over all 10 time dimensions

# Check the cube still has lazy data after defining these operations
print(f"Cube temp_mean_cube has lazy data: {temp_mean_cube.has_lazy_data()}")

# The operations have not been executed yet, and we can visualize them using the task graph
dask_array = temp_mean_cube.core_data()
dask_array.visualize(filename=None)

In [ ]:
# "Touch" the temp_mean_cube data to load it into memory. This will execute the operations shown in the task graph
data = temp_mean_cube.data

# and we can see that the data is no longer lazy
print(f"Cube temp_mean_cube has lazy data: {temp_mean_cube.has_lazy_data()}") # should now be false

# 2. Plotting with iris 

## 2.1 Regional example

In [ ]:
import iris.quickplot as qplt
# The iris quickplot module allows for quick visualisation of data in an iris cube

In [ ]:
# air_temperature example
air_temp_cube = cubes.extract('air_temperature')[0] # Extract the first (and only, in this case) cube with standard_name 'air_temperature'
upward_air_velocity_cube = cubes.extract('upward_air_velocity')[0] # Extract upward_air_velocity cube

qplt.pcolormesh(air_temp_cube[0, :, 64, :])  # Plot the first time step

In [ ]:
# Plot the initial and final states for the central column of the domain

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

# Initial 
plt.subplot(1, 2, 1)
qplt.plot(air_temp_cube[0, :, 64, 64],air_temp_cube[0, :, 64, 64].coord("height")) # Plot the first time step
ax = plt.gca()

# Final
ax = plt.subplot(1, 2, 2)
qplt.plot(air_temp_cube[-1, :, 64, 64],air_temp_cube[-1, :, 64, 64].coord("height")) # Plot the last time step

plt.show()

In [ ]:
# Lets also plot the domain-averaged temperatures we calculated earlier
plt.figure(figsize=(10, 6))
qplt.plot(temp_mean_cube)
plt.show()

In [ ]:
# Animation example

import iris.plot as iplt
import cartopy.crs as ccrs

# Ensure inline animation rendering in Jupyter
plt.rcParams['animation.html'] = 'html5'
from IPython.display import HTML

cube_iter = upward_air_velocity_cube[:,0:70,64,:].slices_over('time')

ani = iplt.animate(cube_iter, qplt.pcolormesh)
# Display inline in notebook
HTML(ani.to_jshtml(fps=5))

## 2.2 Global example

In [ ]:
fname = iris.sample_data_path('air_temp.pp') # Load sample dataset
cubes = iris.load(fname) 
cube = cubes[0] # Extract the first cube (air temperature)
cube

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(1, 2, 1)
qplt.pcolormesh(cube)
ax = plt.gca()
ax.coastlines() # Add coastlines to the plot

ax = plt.subplot(1, 2, 2, projection=ccrs.Orthographic(central_longitude=0, central_latitude=51)) # Create a subplot with an orthographic projection centered on UK
qplt.pcolormesh(cube)
ax.coastlines()

plt.show()